# Three-model precipitation-class violin comparison

A 3x3 figure that includes all three models, with area fraction, greenhouse enhancement factor (GEF), and precipitation efficiency ($\epsilon$) plotted.

In [ ]:
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
from matplotlib import rc
from matplotlib.patches import Patch
import numpy as np
import seaborn as sns
import xarray as xr

In [ ]:
DATA_DIR = Path('data')
FIGURE_DIR = Path('figures')
FIGURE_DIR.mkdir(exist_ok=True)

RCE_PATH = DATA_DIR / 'rce_ctl_violin_days_0001_0021excl.nc'
# RCE_PATH = DATA_DIR / 'rce_ctl_violin_days_0001_0041excl.nc'
# RCE_PATH = DATA_DIR / 'rce_ctl_violin_days_0001_0061excl.nc'
# MPAS_PATH = DATA_DIR / 'ctl_crh_composite.nc' # 0-15N
# MPAS_PATH = DATA_DIR / 'ctl_crh_composite_latmax_10N.nc' # 0-10N
MPAS_PATH = DATA_DIR / 'ctl_crh_composite_lat10_20N.nc' # 10-20N
WRF_PROFILE_GLOB = 'mean_profiles_ctl_haiyan_memb_*.pkl'
WRF_PCLASS_GLOB = 'pclass_ctl_48hrs_haiyan_memb_*.pkl'

CLASS_CODES = [1, 4, 5, 6]
CLASS_KEYS = ['Deep', 'Strat', 'Anvil', 'MCS']
CLASS_LABELS = ['Deep', 'Strat', 'Anvil', 'DSA']
PALETTE = ['teal', 'plum', 'darkorange', 'gold']

## Read and standardize the three model outputs

Each reader returns arrays shaped `(class, sample)`. WRF samples are the 48 matched CTL hours from each of ten members; SAM and MPAS use the samples stored in their derived NetCDF output.

In [ ]:
def _finite_rows(values):
    values = np.asarray(values)
    return [row[np.isfinite(row)] for row in values]


def read_wrf():
    profile_paths = sorted(DATA_DIR.glob(WRF_PROFILE_GLOB))
    pclass_paths = sorted(DATA_DIR.glob(WRF_PCLASS_GLOB))
    if not profile_paths or len(profile_paths) != len(pclass_paths):
        raise FileNotFoundError('Expected matching WRF profile and pclass files')

    pressure = np.arange(1000, 25, -25)
    temperature_factor = (pressure / 1000) ** (287 / 1004)
    dp, gravity, cp, lv = 2500.0, 9.81, 1004.0, 2.5e6
    time_slice = slice(37, 85)  # same t0_tests+1:t1_tests selection as figure2-3.ipynb
    area_members, gef_members, epsilon_members = [], [], []

    for profile_path, pclass_path in zip(profile_paths, pclass_paths):
        with profile_path.open('rb') as file:
            profiles = pickle.load(file)
        with pclass_path.open('rb') as file:
            class_area = np.asarray(pickle.load(file))

        # pclass files already contain the matched 48-hour window.
        area_members.append(100 * class_area[CLASS_CODES])
        member_gef, member_epsilon = [], []
        for key in CLASS_KEYS:
            lw = np.asarray(profiles['RTHRATLW'][key])[time_slice] * temperature_factor
            lw_clear = np.asarray(profiles['RTHRATLWC'][key])[time_slice] * temperature_factor
            # lw_acre = np.sum(cp * (lw - lw_clear), axis=-1) * dp / gravity
            lw_acre = np.asarray(profiles['lwacre'][key])[time_slice]
            rain_flux = np.asarray(profiles['rain'][key])[time_slice] / 3600
            member_gef.append(lw_acre / (lv * rain_flux))
            mu = np.asarray(profiles['vmfu'][key])[time_slice]
            md = np.asarray(profiles['vmfd'][key])[time_slice]
            member_epsilon.append(1 - (-md / mu))
        gef_members.append(member_gef)
        epsilon_members.append(member_epsilon)

    return {
        'area_fraction': np.concatenate(area_members, axis=1),
        'gef': np.concatenate(gef_members, axis=1),
        'epsilon': np.concatenate(epsilon_members, axis=1),
    }


def read_rce():
    with xr.open_dataset(RCE_PATH) as ds:
        return {name: np.stack([ds[name].sel(class_code=code).values.ravel()
                                for code in CLASS_CODES])
                for name in ['area_fraction', 'gef', 'epsilon']}


def read_mpas():
    variables = {
        'area_fraction': 'pclass_area_fraction',
        'gef': 'pclass_gef',
        'epsilon': 'pclass_precip_efficiency',
    }
    with xr.open_dataset(MPAS_PATH) as ds:
        return {name: ds[source].sel(pclass=CLASS_CODES).transpose('pclass', 'time').values
                for name, source in variables.items()}


models = {
    # 'WRF TC': read_wrf(),
    # 'MPAS aquaplanet': read_mpas(),
    # 'SAM RCE': read_rce(),
    'WRF': read_wrf(),
    'MPAS': read_mpas(),
    'SAM': read_rce(),
}
{model: {name: values.shape for name, values in diagnostics.items()}
 for model, diagnostics in models.items()}

## Plot

In [ ]:
font = {'family': 'sans-serif', 'weight': 'normal', 'size': 12}
rc('font', **font)
sns.set_theme(style='ticks', font_scale=1.2, rc={
    'xtick.bottom': True, 'ytick.left': True,
    'axes.spines.right': False, 'axes.spines.top': False,
})
sns.set_palette(PALETTE)

variables = ['area_fraction', 'gef', 'epsilon']
units = ['%', '', '']

## Plot with models as columns

In [ ]:
row_labels = ['Area fraction', 'GEF', r'$\epsilon$']
column_titles = [f'({letter}) {model_name}'
                 for letter, model_name in zip('abc', models)]

fig, axes = plt.subplots(3, 3, figsize=(10, 7.5), layout='constrained', dpi=300,
                         gridspec_kw={'hspace': 0.1})

for row, (variable, row_label, unit) in enumerate(zip(variables, row_labels, units)):
    for column, (model_name, diagnostics) in enumerate(models.items()):
        ax = axes[row, column]
        # sns.violinplot(data=_finite_rows(diagnostics[variable]), width=0.7,
        #                inner='box', ax=ax, legend='full')
        sns.boxplot(data=_finite_rows(diagnostics[variable]), ax=ax,
                    width=0.6, showfliers=True, legend='full',
                    showcaps=False, showmeans=True,
                    meanprops={"marker":".", "markerfacecolor":"white", 
                               "markeredgecolor":"black", "markersize":"6"},
                    flierprops={"marker":".", "markerfacecolor":"black", 
                               "markeredgecolor":"black", "markersize":"2"})
        if row == 0:
            ax.set_title(column_titles[column])
        sns.despine(offset=10, ax=ax, bottom=True)
        if variable == 'area_fraction':
            area_fraction_ticks = [0.5, 1, 2, 5, 10, 20, 50]
            ax.set_yscale('log')
            ax.set_ylim(0.4, 50)
            ax.set_yticks(area_fraction_ticks,
                          labels=[f'{tick:g}' for tick in area_fraction_ticks])
        elif variable == 'gef':
            ax.set_yscale('log')
            ax.set_ylim(0.004, 2)
        elif variable == 'epsilon':
            ax.set_ylim(-0.2, 0.82)
        if ax.get_legend() is not None:
            ax.get_legend().remove()
        if column == 0:
            ax.set_ylabel(unit)
        else:
            ax.set_yticklabels([])
            # ax.spines['left'].set_visible(False)
        ax.set_xticks([])
    axes[row, 0].annotate(row_label, xy=(0, 0.5), xytext=(-54, 0),
                          xycoords='axes fraction', textcoords='offset points',
                          ha='right', va='center', rotation=90)

# handles = [Patch(facecolor=color, edgecolor=color, label=label)
#            for color, label in zip(PALETTE, CLASS_LABELS)]
handles, labels = axes[2,2].get_legend_handles_labels()
axes[-2, -1].legend(handles, CLASS_LABELS, loc='upper right', bbox_to_anchor=(1.6, 0.75),
                    borderaxespad=0, frameon=False)

figure_path = FIGURE_DIR / 'ctl_violin_three_models.pdf'
fig.savefig(figure_path, bbox_inches='tight')
plt.show()
print(figure_path)

In [ ]:
# Mean of the finite values represented by each boxplot
model_width = max(len('Model'), *(len(name) for name in models))
header = f"{'Model':<{model_width}}  " + '  '.join(f'{label:>10}' for label in CLASS_LABELS)

summary_labels = {'area_fraction': 'Area fraction', 'gef': 'GEF', 'epsilon': 'Epsilon'}
for variable, unit in zip(variables, units):
    unit_label = f' ({unit})' if unit else ''
    print(f'{summary_labels[variable]}{unit_label}')
    print(header)
    print('-' * len(header))
    for model_name, diagnostics in models.items():
        means = [np.mean(values) for values in _finite_rows(diagnostics[variable])]
        print(f"{model_name:<{model_width}}  " + '  '.join(f'{mean:10.4f}' for mean in means))
    print()